# Part 02 — Capacity and C-rate

This notebook is the second part of the `battery-core` fundamentals sequence.

**Learning goals**

- distinguish nominal capacity in ampere-hours from current in amperes;
- calculate current from capacity and C-rate;
- calculate the ideal duration associated with a C-rate;
- separate the exact C-rate definition from real-cell voltage-cutoff behavior.

The explanations, examples, code, and exercises here are independently authored for this project.

## 1. Definitions

For nominal capacity $Q_\mathrm{nominal}$ in ampere-hours and current magnitude $I$ in amperes,

$$
C_\mathrm{rate}=\frac{I}{Q_\mathrm{nominal}},
\qquad
I=C_\mathrm{rate}Q_\mathrm{nominal}.
$$

If the full nominal capacity were available at every current, the ideal constant-current duration would be

$$
t_\mathrm{ideal}
=\frac{Q_\mathrm{nominal}}{I}
=\frac{1}{C_\mathrm{rate}}.
$$

The final expression gives time in hours when the numerical C-rate is expressed in reciprocal hours.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from battery_core import (
    c_rate_from_current,
    current_from_c_rate,
    ideal_duration_hours,
)

## 2. A 20 Ah example

The next cell evaluates C/10, 1C, 2C, and 10C using the tested package functions.

In [ ]:
nominal_capacity_ah = 20.0
rates = np.array([0.1, 1.0, 2.0, 10.0])
labels = ["C/10", "1C", "2C", "10C"]

currents_a = current_from_c_rate(nominal_capacity_ah, rates)
durations_h = ideal_duration_hours(rates)

print(f"{'Rate':>6} {'Current [A]':>14} {'Ideal time [h]':>16} {'Ideal time [min]':>18}")
for label, current, hours in zip(labels, currents_a, durations_h):
    print(f"{label:>6} {current:14.1f} {hours:16.3f} {hours * 60.0:18.1f}")

For the 20 Ah example, 1C is 20 A and the ideal duration is 1 hour. At 10C, the current is 200 A and the ideal duration is 0.1 hour, or 6 minutes.

## 3. Current scales linearly with C-rate

For a fixed nominal capacity, doubling the C-rate doubles the current.

In [ ]:
plot_rates = np.logspace(-1, 1, 200)
plot_currents = current_from_c_rate(nominal_capacity_ah, plot_rates)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(plot_rates, plot_currents)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(
    xlabel="C-rate [h⁻¹]",
    ylabel="Current magnitude [A]",
    title="Ideal current for a 20 Ah cell",
)
ax.grid(True, which="both", alpha=0.3)
plt.show()

## 4. Ideal duration is the reciprocal of C-rate

This curve is a definition, not a prediction of a real cell's cutoff time.

In [ ]:
plot_durations_h = ideal_duration_hours(plot_rates)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(plot_rates, plot_durations_h)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(
    xlabel="C-rate [h⁻¹]",
    ylabel="Ideal duration [h]",
    title="Ideal constant-current duration",
)
ax.grid(True, which="both", alpha=0.3)
plt.show()

## 5. Recovering C-rate from a measured current

When a current magnitude and nominal capacity are known, the rate is their ratio.

In [ ]:
current_a = 50.0
capacity_ah = 20.0
rate = c_rate_from_current(current_a, capacity_ah)
print(f"{current_a:.1f} A applied to a {capacity_ah:.1f} Ah cell corresponds to {rate:.2f}C.")

## 6. Why real discharge time can differ

The ideal arithmetic assumes that the entire nominal capacity is available at the selected current. A real cell can reach its lower voltage limit earlier or later than the ideal duration because measured usable capacity depends on test conditions and cell behavior.

Relevant effects include:

- instantaneous ohmic voltage drop;
- slower polarization and transport losses;
- temperature;
- chemistry and electrode design;
- state of health;
- the current profile and voltage limits.

This notebook does **not** fit an empirical rate-capacity law and does not simulate terminal voltage. Those require a defined cell model and validated data.

## 7. Exercises

1. Calculate the current for a 3.2 Ah cell at C/2, 1C, and 3C.
2. Convert the ideal durations for 0.25C, 1.5C, and 5C into minutes.
3. A 5 Ah cell is discharged at 7.5 A. Calculate its C-rate.
4. Explain why the statement “10C always lasts exactly 6 minutes” is not a valid real-cell prediction.
5. Find a public datasheet or dataset and compare one measured discharge duration with the ideal value. Record the temperature, voltage limits, and cell condition before drawing conclusions.